# 🔍 DeepSeek-OCR-2 PDF Parser
### Google Colab Notebook (T4 GPU)

This notebook parses PDF files page-by-page using `deepseek-ai/DeepSeek-OCR-2`.
Each page is rendered to an image via PyMuPDF — no Poppler needed.

> ⚠️ **T4 Note:** bfloat16 is not supported on T4. This notebook uses `float16` automatically.

---
**Steps:**
1. Check GPU
2. Install dependencies
3. Mount Google Drive (optional)
4. Configure & run

## 1. 🖥️ Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # T4 does NOT support bfloat16 — confirm
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f'bfloat16 support: {bf16_ok}  →  will use {"bfloat16" if bf16_ok else "float16"}')

Sat Mar 21 18:52:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 📦 Install Dependencies
This may take ~3–5 minutes on first run.

In [ ]:
# Install model dependencies
!pip install -q \
    transformers==4.46.3 \
    tokenizers==0.20.3 \
    PyMuPDF \
    Pillow \
    numpy \
    einops \
    easydict \
    addict \
    huggingface_hub \
    hf_xet

print('✅ All dependencies installed.')

✅ All dependencies installed.


## 3. 📁 Mount Google Drive (optional)
Skip this cell if you'll upload your PDF directly to Colab.

In [ ]:
USE_GOOGLE_DRIVE = True  # Set to True to mount Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted at /content/drive')
else:
    print('ℹ️  Drive not mounted. Upload your PDF using the cell below.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive


## 4. ⚙️ Configuration
Edit the settings below before running.

In [ ]:
import os

# ── PDF path ──────────────────────────────────────────────────────────────────
# Local upload:  '/content/your_file.pdf'
# Google Drive:  '/content/drive/MyDrive/your_file.pdf'
PDF_PATH   = '/content/drive/MyDrive/tyt-biyoloji.pdf'   # ← CHANGE THIS

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/ocr_biyoloji_ogm_output'

# ── OCR mode ─────────────────────────────────────────────────────────────────
# 'markdown' : structured markdown output  (recommended)
# 'free'     : raw OCR text
OCR_MODE   = 'free'

# ── Page range ───────────────────────────────────────────────────────────────
# None          → process all pages
# (1, 10)       → process pages 1–10 only
PAGE_RANGE = (9,175)

# ── DPI ──────────────────────────────────────────────────────────────────────
# 200 is a good balance. Increase for small text, decrease to save VRAM.
DPI        = 300

# ── Retry empty pages ────────────────────────────────────────────────────────
# Set True to only reprocess pages that produced empty output last run
RETRY_EMPTY = False

# ── Validation ───────────────────────────────────────────────────────────────
assert os.path.exists(PDF_PATH), f'❌ PDF not found: {PDF_PATH}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ PDF       : {PDF_PATH}')
print(f'✅ Output    : {OUTPUT_DIR}')
print(f'✅ Mode      : {OCR_MODE}')
print(f'✅ Pages     : {PAGE_RANGE or "all"}')
print(f'✅ DPI       : {DPI}')

✅ PDF       : /content/drive/MyDrive/tyt-biyoloji.pdf
✅ Output    : /content/drive/MyDrive/ocr_biyoloji_ogm_output
✅ Mode      : free
✅ Pages     : (9, 175)
✅ DPI       : 300


## 5. 🔧 Core Library
Loads the OCR engine. Run once per session.

In [ ]:
import os, sys, json, glob, types, importlib.util, subprocess, traceback
import torch
import fitz
from PIL import Image

fitz.TOOLS.mupdf_display_errors(False)

# ── Flash attention (safe on T4 / any PyTorch 2.x) ───────────────────────────
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ── Constants ─────────────────────────────────────────────────────────────────
MODEL_NAME         = 'deepseek-ai/DeepSeek-OCR-2'
DEFAULT_BASE_SIZE  = 1024
DEFAULT_IMAGE_SIZE = 768

PROMPTS = {
    'free'    : '<image>\nFree OCR. ',
    'markdown': '<image>\n<|grounding|>Convert the document to markdown. ',
}

# Retry strategies: (label, crop_mode, base_size, image_size, dpi)
RETRY_STRATEGIES = [
    ('default',    True,  1024, 768, 200),
    ('no-crop',    False, 1024, 768, 200),
    ('small-tile', True,  768,  512, 200),
    ('low-res',    True,  1024, 768, 150),
]

print('✅ Imports OK.')

✅ Imports OK.


In [ ]:
# ── Package loader (loads model snapshot as importable package) ───────────────

def load_snapshot_as_package(model_dir: str, package_name: str = 'deepseek_ocr2_pkg'):
    if package_name in sys.modules:
        return sys.modules[package_name]

    pkg = types.ModuleType(package_name)
    pkg.__path__    = [model_dir]
    pkg.__package__ = package_name
    pkg.__file__    = os.path.join(model_dir, '__init__.py')
    sys.modules[package_name] = pkg

    for fname in sorted(os.listdir(model_dir)):
        if not fname.endswith('.py'):
            continue
        mod_name  = fname[:-3]
        full_name = f'{package_name}.{mod_name}'
        fpath     = os.path.join(model_dir, fname)
        spec      = importlib.util.spec_from_file_location(
                        full_name, fpath, submodule_search_locations=[])
        module             = importlib.util.module_from_spec(spec)
        module.__package__ = package_name
        sys.modules[full_name] = module
        try:
            spec.loader.exec_module(module)
            setattr(pkg, mod_name, module)
        except ImportError as e:
            missing = str(e).replace("No module named '", '').replace("'", '').split('.')[0]
            print(f'[INFO] Auto-installing missing package: {missing}')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', missing, '-q'])
            try:
                spec2   = importlib.util.spec_from_file_location(
                              full_name, fpath, submodule_search_locations=[])
                module2             = importlib.util.module_from_spec(spec2)
                module2.__package__ = package_name
                sys.modules[full_name] = module2
                spec2.loader.exec_module(module2)
                setattr(pkg, mod_name, module2)
            except Exception as e2:
                sys.modules.pop(full_name, None)
                print(f'[DEBUG] Skipped {fname} after install: {e2}')
        except Exception as e:
            sys.modules.pop(full_name, None)
            print(f'[DEBUG] Skipped {fname}: {e}')

    return pkg


def find_ocr_class(pkg):
    import inspect
    for attr_name in dir(pkg):
        module = getattr(pkg, attr_name, None)
        if not isinstance(module, types.ModuleType):
            continue
        for cls_name, cls in inspect.getmembers(module, inspect.isclass):
            if hasattr(cls, 'infer') and hasattr(cls, 'from_pretrained'):
                print(f'[INFO] Found OCR class : {cls_name}  (in {attr_name}.py)')
                return cls
    return None

print('✅ Package loader defined.')

✅ Package loader defined.


In [ ]:
# ── Model loader ─────────────────────────────────────────────────────────────
# T4 FIX: uses float16 instead of bfloat16 (T4 does not support bfloat16)

_model     = None
_tokenizer = None
_device    = None


def load_model(model_name: str = MODEL_NAME):
    global _model, _tokenizer, _device

    if _model is not None:
        return _model, _tokenizer

    _device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── T4 FIX ────────────────────────────────────────────────────────────────
    # bfloat16 requires Ampere+ (A100, 3090, etc.).
    # T4 is Turing-generation → use float16 instead.
    bf16_supported = _device == 'cuda' and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_supported else torch.float16
    # ─────────────────────────────────────────────────────────────────────────

    print(f'[INFO] Loading model : {model_name}')
    print(f'[INFO] Device        : {_device.upper()}  |  dtype: {dtype}')

    from huggingface_hub import snapshot_download
    from transformers import AutoTokenizer

    model_dir  = snapshot_download(repo_id=model_name)
    print(f'[INFO] Snapshot      : {model_dir}')

    py_files = [f for f in os.listdir(model_dir) if f.endswith('.py')]
    print(f'[INFO] Python files  : {py_files}')

    pkg        = load_snapshot_as_package(model_dir, 'deepseek_ocr2_pkg')
    _tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    ModelClass = find_ocr_class(pkg)

    if ModelClass is not None:
        print(f'[INFO] Loading with  : {ModelClass.__name__}.from_pretrained()')
        _model = ModelClass.from_pretrained(
            model_dir,
            torch_dtype=dtype,
            trust_remote_code=True,
        )
    else:
        # Direct-file fallback
        print('[WARN] Package scan failed — trying direct file import.')
        import importlib.util as _ilu, inspect as _inspect
        ocr2_path = os.path.join(model_dir, 'modeling_deepseekocr2.py')
        if os.path.exists(ocr2_path):
            spec = _ilu.spec_from_file_location('modeling_deepseekocr2', ocr2_path)
            mod  = _ilu.module_from_spec(spec)
            mod.__package__ = 'deepseek_ocr2_pkg'
            sys.modules['modeling_deepseekocr2'] = mod
            sys.modules['deepseek_ocr2_pkg.modeling_deepseekocr2'] = mod
            spec.loader.exec_module(mod)
            for cname, cls in _inspect.getmembers(mod, _inspect.isclass):
                if hasattr(cls, 'infer') and hasattr(cls, 'from_pretrained'):
                    ModelClass = cls
                    print(f'[INFO] Direct import found: {cname}')
                    break

        if ModelClass is not None:
            _model = ModelClass.from_pretrained(
                model_dir, torch_dtype=dtype, trust_remote_code=True,
            )
        else:
            print('[WARN] Falling back to AutoModel.')
            from transformers import AutoModel
            _model = AutoModel.from_pretrained(
                model_dir,
                _attn_implementation='eager',
                trust_remote_code=True,
                use_safetensors=True,
                torch_dtype=dtype,
            )

    _model = _model.eval().to(_device)

    # torch.compile — ~20 % speed boost after first-page warm-up
    try:
        _model = torch.compile(_model, mode='reduce-overhead')
        print('[INFO] torch.compile  : enabled (first page will be slower).')
    except Exception as e:
        print(f'[WARN] torch.compile skipped: {e}')

    print(f'[INFO] Model ready on {_device.upper()}.\n')
    return _model, _tokenizer

print('✅ Model loader defined.')

✅ Model loader defined.


In [ ]:
# ── PDF rendering helpers ─────────────────────────────────────────────────────

def pdf_to_images(pdf_path: str, dpi: int = 200, page_range=None):
    print(f'[INFO] Opening PDF : {pdf_path}')
    doc   = fitz.open(pdf_path)
    total = len(doc)
    print(f'[INFO] Total pages : {total}')

    if page_range:
        first = max(1, page_range[0]) - 1
        last  = min(total, page_range[1]) - 1
    else:
        first, last = 0, total - 1

    zoom   = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for idx in range(first, last + 1):
        page = doc[idx]
        pix  = page.get_pixmap(matrix=matrix, colorspace=fitz.csRGB)
        img  = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
        pages.append((idx + 1, img))
        print(f'[INFO]   Rendered page {idx + 1}  ({pix.width}x{pix.height} px)')
    doc.close()
    print(f'[INFO] {len(pages)} page(s) ready.\n')
    return pages


def render_single_page(pdf_path: str, page_num: int, dpi: int = 200) -> Image.Image:
    doc    = fitz.open(pdf_path)
    page   = doc[page_num - 1]
    zoom   = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pix    = page.get_pixmap(matrix=matrix, colorspace=fitz.csRGB)
    img    = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
    doc.close()
    return img


# ── .mmd file reader ──────────────────────────────────────────────────────────

def read_mmd_output(output_path: str, img_stem: str) -> str:
    exact = os.path.join(output_path, f'{img_stem}.mmd')
    if os.path.exists(exact):
        with open(exact, encoding='utf-8') as f:
            return f.read().strip()
    mmd_files = sorted(glob.glob(os.path.join(output_path, '*.mmd')),
                       key=os.path.getmtime, reverse=True)
    if mmd_files:
        with open(mmd_files[0], encoding='utf-8') as f:
            return f.read().strip()
    return ''


# ── Single inference call ─────────────────────────────────────────────────────

def run_infer(
    image: Image.Image,
    output_path: str,
    prompt: str,
    img_stem: str,
    base_size: int  = DEFAULT_BASE_SIZE,
    image_size: int = DEFAULT_IMAGE_SIZE,
    crop_mode: bool = True,
) -> str:
    model, tokenizer = load_model()
    tmp_img_path = os.path.join(output_path, f'{img_stem}.png')
    image.save(tmp_img_path, format='PNG')
    try:
        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=tmp_img_path,
            output_path=output_path,
            base_size=base_size,
            image_size=image_size,
            crop_mode=crop_mode,
            save_results=True,
            test_compress=True,
        )
    except torch.cuda.OutOfMemoryError:
        print('\n[WARN] CUDA OOM — clearing cache.')
        torch.cuda.empty_cache()
        return ''
    except Exception as exc:
        print(f'\n[WARN] infer() raised: {exc}')
        return ''
    finally:
        if os.path.exists(tmp_img_path):
            os.remove(tmp_img_path)

    text = ''
    if result is not None:
        if isinstance(result, dict):
            text = result.get('text', result.get('output', ''))
        elif isinstance(result, str):
            text = result.strip()
    if not text:
        text = read_mmd_output(output_path, img_stem)
    return text


# ── Split-half strategy ───────────────────────────────────────────────────────

def ocr_split_halves(image: Image.Image, output_path: str, prompt: str, page_num: int) -> str:
    w, h   = image.size
    top    = image.crop((0, 0, w, h // 2))
    bottom = image.crop((0, h // 2, w, h))
    top_dir = os.path.join(output_path, 'split_top')
    bot_dir = os.path.join(output_path, 'split_bottom')
    os.makedirs(top_dir, exist_ok=True)
    os.makedirs(bot_dir, exist_ok=True)
    print(f'\n         [split] top half ...', end=' ', flush=True)
    top_text = run_infer(top, top_dir, prompt, f'page_{page_num:04d}_top')
    print('OK' if top_text else 'empty')
    print(f'         [split] bottom half ...', end=' ', flush=True)
    bot_text = run_infer(bottom, bot_dir, prompt, f'page_{page_num:04d}_bottom')
    print('OK' if bot_text else 'empty')
    return '\n\n'.join(t for t in [top_text, bot_text] if t)


# ── Per-page OCR with retry ───────────────────────────────────────────────────

def ocr_page_with_retry(image, output_path, prompt, page_num, pdf_path):
    for label, crop_mode, base_size, image_size, dpi in RETRY_STRATEGIES:
        print(f'\n         [strategy: {label}] ...', end=' ', flush=True)
        img  = render_single_page(pdf_path, page_num, dpi=dpi) if dpi != 200 else image
        stem = f'page_{page_num:04d}_{label}'
        text = run_infer(img, output_path, prompt, stem, base_size, image_size, crop_mode)
        if text:
            print(f'OK  ({len(text)} chars)')
            return text, label
        print('empty — trying next strategy')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f'\n         [strategy: split-halves] ...', end=' ', flush=True)
    text = ocr_split_halves(image, output_path, prompt, page_num)
    return (text, 'split-halves') if text else ('', 'all-failed')


print('✅ All helper functions defined.')

✅ All helper functions defined.


## 6. 🤖 Load Model
Downloads ~5 GB on first run. Subsequent runs use the cached copy.

In [ ]:
model, tokenizer = load_model(MODEL_NAME)
print('\n✅ Model loaded and ready.')

[INFO] Loading model : deepseek-ai/DeepSeek-OCR-2
[INFO] Device        : CUDA  |  dtype: torch.bfloat16


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

[INFO] Snapshot      : /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-OCR-2/snapshots/aaa02f3811945a91062062994c5c4a3f4c0af2b0
[INFO] Python files  : ['modeling_deepseekocr2.py', 'conversation.py', 'modeling_deepseekv2.py', 'configuration_deepseek_v2.py', 'deepencoderv2.py']


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
DeepseekV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).

[INFO] Found OCR class : DeepseekOCR2ForCausalLM  (in modeling_deepseekocr2.py)
[INFO] Loading with  : DeepseekOCR2ForCausalLM.from_pretrained()
[INFO] torch.compile  : enabled (first page will be slower).
[INFO] Model ready on CUDA.


✅ Model loaded and ready.


## 7. 🚀 Run OCR

In [ ]:
def parse_pdf(pdf_path, output_dir, mode='markdown', dpi=200, page_range=None, retry_empty=False):
    pdf_path   = os.path.abspath(pdf_path)
    output_dir = os.path.abspath(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    prompt = PROMPTS.get(mode, PROMPTS['markdown'])
    print(f'[INFO] OCR mode : "{mode}"')
    print(f'[INFO] Prompt   : {prompt!r}')
    print(f'[INFO] Model    : {MODEL_NAME}\n')

    # --retry-empty mode
    if retry_empty:
        page_images = []
        for page_dir in sorted(glob.glob(os.path.join(output_dir, 'page_*'))):
            page_num = int(os.path.basename(page_dir).split('_')[1])
            txt_file = os.path.join(page_dir, f'page_{page_num:04d}.txt')
            if os.path.exists(txt_file):
                content = open(txt_file, encoding='utf-8').read().strip()
                if not content:
                    img = render_single_page(pdf_path, page_num, dpi=dpi)
                    page_images.append((page_num, img))
                    print(f'[INFO] Queued empty page {page_num} for retry.')
        if not page_images:
            print('[INFO] No empty pages found.')
            return {}
    else:
        page_images = pdf_to_images(pdf_path, dpi=dpi, page_range=page_range)

    total_pages     = len(page_images)
    results         = {}
    strategies_used = {}

    for i, (page_num, image) in enumerate(page_images, 1):
        page_out_dir = os.path.join(output_dir, f'page_{page_num:04d}')
        os.makedirs(page_out_dir, exist_ok=True)

        print(f'[{i}/{total_pages}] OCR page {page_num} ...', end=' ', flush=True)

        try:
            stem = f'page_{page_num:04d}_default'
            text = run_infer(image, page_out_dir, prompt, stem)

            if text:
                strategy = 'default'
                print(f'OK  ({len(text)} chars)  |  {text[:60].replace(chr(10), " ")!r}')
            else:
                print('empty — retrying ...')
                text, strategy = ocr_page_with_retry(
                    image, page_out_dir, prompt, page_num, pdf_path
                )

            results[page_num]         = text
            strategies_used[page_num] = strategy

            txt_file = os.path.join(page_out_dir, f'page_{page_num:04d}.txt')
            with open(txt_file, 'w', encoding='utf-8') as f:
                f.write(text)

            if not text:
                print(f'         [WARN] Page {page_num} empty after all strategies.')

        except Exception as exc:
            print(f'\n[ERROR] Page {page_num}: {exc}')
            traceback.print_exc()
            results[page_num]         = f'[ERROR: {exc}]'
            strategies_used[page_num] = 'error'

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Save combined outputs ─────────────────────────────────────────────────
    combined_txt  = os.path.join(output_dir, 'full_text.txt')
    combined_mmd  = os.path.join(output_dir, 'full_text.mmd')
    combined_json = os.path.join(output_dir, 'full_text.json')

    with open(combined_txt, 'w', encoding='utf-8') as f:
        for p in sorted(results):
            f.write(f"\n{'='*60}\nPAGE {p}  [strategy: {strategies_used.get(p,'?')}]\n{'='*60}\n\n")
            f.write(results[p])
            f.write('\n')

    with open(combined_mmd, 'w', encoding='utf-8') as f:
        for p in sorted(results):
            f.write(f'\n<!-- PAGE {p} -->\n\n')
            f.write(results[p])
            f.write('\n')

    with open(combined_json, 'w', encoding='utf-8') as f:
        json.dump(
            {str(k): {'text': v, 'strategy': strategies_used.get(k, '?')}
             for k, v in sorted(results.items())},
            f, ensure_ascii=False, indent=2,
        )

    empty_pages  = [p for p, t in results.items() if not t or t.startswith('[ERROR')]
    filled_pages = [p for p, t in results.items() if t and not t.startswith('[ERROR')]

    print(f"\n{'='*60}")
    print('[INFO] SUMMARY')
    print(f"{'='*60}")
    print(f'  Model            : {MODEL_NAME}')
    print(f'  Successful pages : {filled_pages}')
    print(f'  Empty/failed     : {empty_pages}')
    print(f'  Strategies used  : {strategies_used}')
    print(f'  Combined text    : {combined_txt}')
    print(f'  Combined mmd     : {combined_mmd}')
    print(f'  Combined json    : {combined_json}')
    if empty_pages:
        print('\n  Tip: set RETRY_EMPTY=True and re-run the OCR cell to retry failed pages.')
    return results


# ── Execute ───────────────────────────────────────────────────────────────────
results = parse_pdf(
    pdf_path    = PDF_PATH,
    output_dir  = OUTPUT_DIR,
    mode        = OCR_MODE,
    dpi         = DPI,
    page_range  = PAGE_RANGE,
    retry_empty = RETRY_EMPTY,
)

[INFO] OCR mode : "free"
[INFO] Prompt   : '<image>\nFree OCR. '
[INFO] Model    : deepseek-ai/DeepSeek-OCR-2

[INFO] Opening PDF : /content/drive/MyDrive/tyt-biyoloji.pdf
[INFO] Total pages : 174
[INFO]   Rendered page 9  (2304x3249 px)
[INFO]   Rendered page 10  (2304x3249 px)
[INFO]   Rendered page 11  (2304x3249 px)
[INFO]   Rendered page 12  (2304x3249 px)
[INFO]   Rendered page 13  (2304x3249 px)
[INFO]   Rendered page 14  (2304x3249 px)
[INFO]   Rendered page 15  (2304x3249 px)
[INFO]   Rendered page 16  (2304x3249 px)
[INFO]   Rendered page 17  (2304x3249 px)
[INFO]   Rendered page 18  (2304x3249 px)
[INFO]   Rendered page 19  (2304x3249 px)
[INFO]   Rendered page 20  (2304x3249 px)
[INFO]   Rendered page 21  (2304x3249 px)
[INFO]   Rendered page 22  (2304x3249 px)
[INFO]   Rendered page 23  (2304x3249 px)
[INFO]   Rendered page 24  (2304x3249 px)
[INFO]   Rendered page 25  (2304x3249 px)
[INFO]   Rendered page 26  (2304x3249 px)
[INFO]   Rendered page 27  (2304x3249 px)
[INFO]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Cal

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
# İÇİNDEKİLER

## Canlıların Ortak Özellikleri
Canlıların Ortak Özellikleri ...... 9

## Canlıların Yapısında Bulunan İnorganik Bileşikler
İnorganik Bileşiklerin Genel Özellikleri ve Canlılar İçin Önemi ...... 16

## Canlıların Yapısında Bulunan Organik Bileşikler
- Organik Bileşiklerin Genel Özellikleri, Karbonhidratlar ...... 20  
- Lipitler ...... 25  
- Proteinler ...... 28  
- Enzimler ...... 31  
- Vitaminler ve Hormonlar ...... 36  
- Nükleik Asitler ...... 38  
- ATP (Adenozin trifosfat) ...... 42  
- Sağlıklı Beslenme ...... 44  

## Hücresel Yapılar ve Görevleri
- Hücre Teorisi, Prokaryot ve Ökaryot Hücre Yapısı ...... 46  
- Hücre Zarı, Hücre Duvarı, Sitoplazma, Çekirdek ...... 50  
- Ribozom, Endoplazmik Retikulum, Golgi Aygıtı, Lizozom, Kofül ...... 54  
- Peroksizom, Sentrozom, Hücre İskeleti ...... 59  
- Mitokondri ve Plastitler ...... 61  

## Hücre Zarından Madde Geçişleri
- Hücre Zarından Madde Ge

image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

OK  (1525 chars)  |  '# İÇİNDEKİLER  ## Canlıların Ortak Özellikleri Canlıların Or'
[2/166] OCR page 10 ... 

...

image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

OK  (2671 chars)  |  '# İNORGANİK BİLEŞİKLERİN GENEL ÖZELLİKLERİ VE CANLILAR İÇİN '
[13/166] OCR page 21 ... 


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
# İNORGANİK BİLEŞİKLERİN GENEL ÖZELLİKLERİ VE CANLILAR İÇİN 

KeyboardInterrupt: 

## 8. 👁️ Preview Results

In [ ]:
from IPython.display import Markdown, display

if results:
    first_page = sorted(results.keys())[0]
    print(f'=== Page {first_page} (first 2000 chars) ===')
    text = results[first_page]
    if OCR_MODE == 'markdown':
        display(Markdown(text[:2000]))
    else:
        print(text[:2000])
else:
    print('No results to preview.')

## 9. 📥 Download Output Files

In [ ]:
from google.colab import files
import zipfile

# Zip the entire output directory and download
zip_path = '/content/ocr_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, OUTPUT_DIR)
            zf.write(fpath, arcname)

print(f'✅ Zipped output to {zip_path}')
files.download(zip_path)

# Or download individual combined files:
# files.download(os.path.join(OUTPUT_DIR, 'full_text.txt'))
# files.download(os.path.join(OUTPUT_DIR, 'full_text.mmd'))
# files.download(os.path.join(OUTPUT_DIR, 'full_text.json'))

## 10. 🔄 Retry Empty Pages (optional)
Run this cell if some pages came back empty.

In [ ]:
retry_results = parse_pdf(
    pdf_path    = PDF_PATH,
    output_dir  = OUTPUT_DIR,
    mode        = OCR_MODE,
    dpi         = DPI,
    retry_empty = True,   # ← only reprocesses empty pages
)
print('Retry complete.')